# Week 4 (Continued) Lab: Robust Device Toolkit
NET2008 DevOps · Algonquin College

This lab is graded (1% of final grade). It applies exceptions, file handling, modules, OOP, and decorators from this week's lecture to a larger, independent problem: turning Hands-On into a toolkit that survives bad data and remembers its work between runs.

**How this notebook is organized:** each numbered section below has a markdown cell explaining what to build, followed by a code cell with `# your code here` where you fill it in. Work top to bottom — later sections depend on the classes and functions you build in earlier ones. Run each cell as you finish it so you catch mistakes early, instead of writing everything and running it all at the end.

**Before you start:** re-read the Week 4 lecture notebook's sections on classes/inheritance, exceptions, and decorators if any of the terms below are unfamiliar.


## 1. Build the `devices` module
Complete `devices.py` below by filling in every `# your code here` spot. Do not remove or rename anything else in the file — the rest of the notebook imports this exact module and expects these exact names.

**What each piece does:**
- `Device` is the base class every device inherits from. It stores `hostname` and `ip`.
- `summary()` builds a one-line human-readable string, e.g. for `Device("SW-CORE-01", "192.168.1.10")` it should return exactly `"SW-CORE-01 (192.168.1.10)"`.
- `__str__` is what runs when you `print(some_device)` or put it in an f-string — just return `self.summary()`.
- `__repr__` is what you see when you inspect a device in a list or at the REPL. Return it in constructor-call form, e.g. `"Device('SW-CORE-01', '192.168.1.10')"` — note the quotes around the string arguments.
- `__eq__` defines what `==` means for two devices. Two devices are equal if they have the same `hostname` (ignore `ip` and type).
- `to_dict()` is already written for you — you'll use it in section 7 to save devices to a file. Look at it now so you understand the shape of data you're working with later.
- `Router` and `Switch` are subclasses of `Device`. Each adds one attribute in `__init__` (call `super().__init__(hostname, ip)` first, then store the new attribute) and overrides `summary()` to call the parent's summary and append extra text:
  - `Router("RTR-CORE-01", "192.168.1.1", "OSPF").summary()` → `"RTR-CORE-01 (192.168.1.1): running OSPF"`
  - `Switch("SW-ACCESS-01", "192.168.1.2", 24).summary()` → `"SW-ACCESS-01 (192.168.1.2): 24 ports"`

**Why `%%writefile`:** the cell below writes `devices.py` to disk as a real, separate module file rather than just defining classes in the notebook. That's why cell 2 (below) starts with `importlib.reload(devices)` — after you edit and re-run this cell, the notebook needs to be told to pick up the new file.


In [ ]:
%%writefile devices.py
"""Device data model for the Week 4 lab."""


class DeviceUnreachableError(Exception):
    """Raised when a device check fails."""
    pass


class Device:
    def __init__(self, hostname, ip):
        self.hostname = hostname
        self.ip = ip

    def summary(self):
        # return a string like "SW-CORE-01 (192.168.1.10)"
        # your code here
        pass

    def __str__(self):
        # reuse summary() here
        # your code here
        pass

    def __repr__(self):
        # return something like Device('SW-CORE-01', '192.168.1.10')
        # your code here
        pass

    def __eq__(self, other):
        # two devices are equal if they have the same hostname
        # your code here
        pass

    def to_dict(self):
        return {"type": type(self).__name__, "hostname": self.hostname, "ip": self.ip}


class Router(Device):
    def __init__(self, hostname, ip, routing_protocol):
        # call super().__init__(...), then store routing_protocol
        # your code here
        pass

    def summary(self):
        # extend the base summary with ": running {routing_protocol}"
        # your code here
        pass


class Switch(Device):
    def __init__(self, hostname, ip, port_count):
        # call super().__init__(...), then store port_count
        # your code here
        pass

    def summary(self):
        # extend the base summary with ": {port_count} ports"
        # your code here
        pass


## 2. Build a 5-device inventory
Import your module, then create a list of 5 devices: a mix of at least one `Router` and one `Switch` (the other 3 can be any mix of the two).

This is plain object construction — no loops or helper functions needed, just 5 lines inside the list. Use made-up hostnames and IPs; they don't need to be real.


In [ ]:
import devices
import importlib
importlib.reload(devices)  # picks up your edits above without restarting the kernel

inventory = [
    # Add 5 Router / Switch instances here. Example of the syntax (replace with your own):
    # devices.Router("RTR-CORE-01", "192.168.1.1", "OSPF"),
    # devices.Switch("SW-ACCESS-01", "192.168.1.2", 24),
]


## 3. Simulate device checks
This helper is provided — it stands in for a real network call, and fails about 30% of the time by raising `devices.DeviceUnreachableError`. **Do not modify this cell.**

Note that when it succeeds, it returns `device.summary()` — so if section 1's `summary()` isn't working yet, this will error out even on success. Get section 1 fully working first.


In [ ]:
import random

def simulate_check(device):
    """Pretend to contact a device. Raises DeviceUnreachableError ~30% of the time."""
    if random.random() < 0.3:
        raise devices.DeviceUnreachableError(f"{device.hostname} did not respond")
    return device.summary()


## 4. Write a `@retry` decorator
Write `retry(times)`, a **decorator factory** (a function that returns a decorator - like the one from the lecture, this is why there are two levels of nested function below: `retry` → `decorator` → `wrapper`).

`wrapper` should call the wrapped function up to `times` times. Each time it raises `devices.DeviceUnreachableError`, catch it and try again; if the very last attempt also fails, let that exception propagate instead of swallowing it. If any attempt succeeds, return its result immediately without trying again. Use `functools.wraps` on `wrapper` so the wrapped function keeps its original name (already set up for you below).

**Worked example of the behaviour you're building**, with `times=3`:
- Attempt 1 fails, attempt 2 fails, attempt 3 succeeds → `wrapper` returns attempt 3's result. No exception is raised.
- All 3 attempts fail → `wrapper` raises `devices.DeviceUnreachableError` (from the 3rd attempt) so the caller sees the failure.


In [ ]:
import functools

def retry(times=3):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # your code here: try func(*args, **kwargs) up to `times` times,
            # catching devices.DeviceUnreachableError; re-raise after the last attempt
            pass
        return wrapper
    return decorator


## 5. Write a `@logged` decorator
Write `logged`, a decorator with **no arguments** (a single level of nesting: `logged` → `wrapper`, unlike `retry` above). It should print `"CALL <function name>(<args>)"` immediately before running the wrapped function, then print `"RESULT: <value>"` immediately after, then return that value.

`check_device` below applies both decorators. **Decorator order matters here:** `@logged` is written directly above the function (applied first, so it's the innermost wrapper) and `@retry(times=3)` is written above that (applied second, so it's the outermost wrapper). Because retry is on the outside, every individual attempt — including ones that fail and get retried — passes through `logged` and gets its own `CALL` / `RESULT` line, so you can see exactly what happened on each attempt.

**Example output** for one call to `check_device(router1)` where the first attempt fails and the second succeeds:
```
CALL check_device(RTR-CORE-01 (192.168.1.1): running OSPF)
CALL check_device(RTR-CORE-01 (192.168.1.1): running OSPF)
RESULT: RTR-CORE-01 (192.168.1.1): running OSPF
```
(Two `CALL` lines for the two attempts, but only one `RESULT` line — the failed attempt raised an exception instead of returning a result.)


In [ ]:
def logged(func):
    # your code here
    pass


@retry(times=3)
@logged
def check_device(device):
    return simulate_check(device)


## 6. Run the checks with graceful degradation
Loop over `inventory`. For each device, call `check_device(device)` inside a `try`/`except`. If it ultimately raises `devices.DeviceUnreachableError` even after retries, catch it, print `"UNREACHABLE: <hostname>"`, and **keep going** — one unreachable device must not stop the loop from checking the rest.

If `check_device` succeeds, print its return value (the device's summary).

**Example output** for a 5-device inventory where one device stays unreachable through all 3 retries:
```
SW-CORE-01 (192.168.1.10): 24 ports
RTR-CORE-01 (192.168.1.1): running OSPF
UNREACHABLE: SW-ACCESS-02
SW-ACCESS-03 (192.168.1.13): 48 ports
RTR-EDGE-01 (192.168.1.20): running BGP
```
(Your run will also show the `CALL`/`RESULT` logging lines from section 5 interleaved with this — that's expected.)


In [ ]:
# your loop here


## 7. Save and reload the inventory
Using `pathlib`:
1. Create an `inventory/` folder next to the notebook (`Path("inventory").mkdir(exist_ok=True)` — `exist_ok=True` means it won't error if the folder already exists from a previous run).
2. Build a list of dicts from your inventory using each device's `to_dict()` (already written for you in section 1).
3. Write that list to `inventory/inventory.json` using `json.dumps(..., indent=2)` for readable formatting, then write the resulting string to the file.
4. Read the file back (`json.loads` on its contents), and print how many devices were reloaded.

**Example output:**
```
Reloaded 5 devices from inventory.json
```


In [ ]:
import json
from pathlib import Path

# your code here


**Checkpoint:** before submitting, use the Jupyter menu to **Restart & Run All** and confirm there are no errors (no red tracebacks) from top to bottom. A device being reported `UNREACHABLE` is expected behavior, not an error — the notebook itself must not crash. This confirms your exceptions, decorators, module, and file handling all work together.

Once it runs clean, follow the renaming and upload steps on the **Week 4 Lab Submission** page before submitting.
